# M1 Colab Smoke Test

This notebook checks the first implementation milestone from `PLAN.md`:

- 1-bit store-and-recall batches with inputs `[value, store, recall]`
- `StackedRNN` forward passes for `n_ff = 0, 1, 2`
- the repo's M1 pytest checks
- a tiny BPTT training smoke test showing `n_ff=0` can learn above chance

Before running this in Colab, push your local M1 branch to GitHub and set `BRANCH` below to that branch name.

In [ ]:
REPO_URL = "https://github.com/simonpeter02/e-prop-in-deep-networks.git"
BRANCH = "main"  # change this to your M1 branch before running in Colab
REPO_DIR = "e-prop-in-deep-networks"

import os
import subprocess
import sys
from pathlib import Path

os.chdir("/content")
repo_path = Path(REPO_DIR)

if repo_path.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=repo_path, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_path, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo_path, check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)

os.chdir(repo_path)
print("Working directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

## Install Dependencies

Colab usually already has PyTorch installed, but this installs whatever the repo declares in `requirements.txt`.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device for M1 checks: CPU")

## Run Repo Tests

These are the committed M1 checks in `tests/test_m1_scaffold.py`.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m1_scaffold.py"], check=True)

## Explicit Shape And Forward Checks

In [ ]:
from tasks.store_and_recall import generate_batch, masked_cross_entropy, task_accuracy
from models.stacked_rnn import StackedRNN

inputs, targets, recall_mask = generate_batch(batch_size=8, delay=5, seed=123)
print("inputs:", tuple(inputs.shape))
print("targets:", tuple(targets.shape))
print("recall_mask:", tuple(recall_mask.shape))
print("mask count per trial:", recall_mask.sum(dim=0).tolist())

assert inputs.shape == (7, 8, 3)
assert targets.shape == (7, 8)
assert recall_mask.shape == (7, 8)
assert torch.all(recall_mask.sum(dim=0) == 1)

for n_ff in (0, 1, 2):
    model = StackedRNN(n_in=3, n_rec=16, n_out=2, n_ff=n_ff)
    logits, states = model(inputs)
    loss = masked_cross_entropy(logits, targets, recall_mask)
    acc = task_accuracy(logits, targets, recall_mask)
    print(f"n_ff={n_ff}: logits={tuple(logits.shape)}, states_per_step={len(states[0])}, loss={loss.item():.4f}, acc={acc:.3f}")
    assert logits.shape == (7, 8, 2)
    assert len(states) == 8
    assert len(states[0]) == 1 + n_ff
    assert torch.isfinite(loss)

print("Explicit M1 checks passed.")

## Tiny BPTT Training Smoke Test

This is intentionally small. It only checks that the M1 task/model scaffold can learn with exact autograd on a flat recurrent net (`n_ff=0`). It is not a final experiment.

In [ ]:
torch.manual_seed(0)

model = StackedRNN(n_in=3, n_rec=32, n_out=2, n_ff=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

eval_inputs, eval_targets, eval_mask = generate_batch(batch_size=256, delay=5, seed=999)
with torch.no_grad():
    start_acc = task_accuracy(model(eval_inputs)[0], eval_targets, eval_mask)

for step in range(300):
    train_inputs, train_targets, train_mask = generate_batch(batch_size=64, delay=5)
    optimizer.zero_grad()
    logits, _ = model(train_inputs)
    loss = masked_cross_entropy(logits, train_targets, train_mask)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        with torch.no_grad():
            acc = task_accuracy(model(eval_inputs)[0], eval_targets, eval_mask)
        print(f"step={step:03d} loss={loss.item():.4f} eval_acc={acc:.3f}")

with torch.no_grad():
    end_acc = task_accuracy(model(eval_inputs)[0], eval_targets, eval_mask)

print(f"start_acc={start_acc:.3f}, end_acc={end_acc:.3f}")
assert end_acc >= 0.80, "BPTT smoke test did not learn clearly above chance"
assert end_acc - start_acc >= 0.20, "BPTT smoke test did not improve enough"
print("M1 BPTT smoke test passed.")

## Expected Result

A passing run should show:

- `pytest` passing for `tests/test_m1_scaffold.py`
- forward passes for `n_ff=0`, `n_ff=1`, and `n_ff=2`
- final BPTT smoke-test accuracy well above chance, usually near `1.0`